In [0]:
%sql
CREATE TABLE IF NOT EXISTS de_learning_workspace.silver.inventory
(
    inventory_id INT,
    product_id INT,

    warehouse_location STRING,

    available_quantity INT,
    reserved_quantity INT,

    inventory_status STRING,

    updated_at TIMESTAMP,

    _load_timestamp TIMESTAMP,
    _source_system STRING,

    _silver_processed_timestamp TIMESTAMP
)
USING DELTA;

In [0]:
from pyspark.sql.functions import *

bronze_df = spark.table("de_learning_workspace.bronze.inventory")

silver_df = (
    bronze_df

    .dropDuplicates(["inventory_id"])

    .withColumn(
        "warehouse_location",
        trim(col("warehouse_location"))
    )

    .withColumn(
        "inventory_status",
        initcap(trim(col("inventory_status")))
    )

    .withColumn(
        "available_quantity",
        coalesce(col("available_quantity"), lit(0))
    )

    .withColumn(
        "reserved_quantity",
        coalesce(col("reserved_quantity"), lit(0))
    )

    .withColumn(
        "_silver_processed_timestamp",
        current_timestamp()
    )
)

(
    silver_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("de_learning_workspace.silver.inventory")
)